### basic quantization

In [1]:
import torch
from torch import nn

from torchvision.models import vgg16

import copy

In [2]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")
print(f"Using device: {device}")


Using device: cpu


In [3]:

MODEL_PATH="runs/vgg16-fp32-baseline/best.pt"
NUM_LABELS=100



train_state_dict = torch.load(MODEL_PATH, map_location=device, weights_only=True)

model = vgg16(weights=None).to(device)
#reduces final classifier to 100 since it defaults to 1000
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, NUM_LABELS) # type: ignore

model.load_state_dict(train_state_dict["model"])
print(model)

# checkpoint = {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
#               "scheduler": scheduler.state_dict(), "epoch": epoch + 1, "best_top1": best_top1,
#               "config": config, "classes": classes, "history": history, "environment": environment,
#               "python_rng": random.getstate(), "torch_rng": torch.get_rng_state(),
#               "loader_rng": generator.get_state(),
#               "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else []}


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [4]:
from torchvision import datasets
from torchvision import transforms

import torchao

from torch.utils.data import DataLoader, Subset
import random



DATA_ROOT_PATH="/home/mrcorcoran/courses/ece690d/project/data/imagenet"

transform = transforms.Compose([
    transforms.Resize(round(224 * 256 / 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])
print()
full_data = datasets.CIFAR100(root=DATA_ROOT_PATH, train=True, transform=transform )


def quantize_calib(model:nn.Sequential, data, samplesize:int=100):

    calibration_subset = Subset(dataset=data, 
                                indices=random.sample(range(0, 101), k=samplesize)
                                )
    calibration_loader = DataLoader(calibration_subset, batch_size=4, shuffle=False)
    model.eval()
    with torch.no_grad():
        # Typically, 100-500 images (a few batches) are sufficient for calibration
        for images, _ in calibration_loader:
            images = images.to(device)
            model(images)

W1104 18:34:50.400000 418079 torch/utils/_pytree.py:646] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [5]:
# Static post-training INT8 quantization for an x86 CPU (TorchAO PT2E).
# Requires compatible PyTorch (>= 2.6) and TorchAO versions.
import copy
import os

from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
from torchao.quantization.pt2e.quantizer.x86_inductor_quantizer import (
    X86InductorQuantizer,
    get_default_x86_inductor_quantization_config,
)

# Match the checkpoint's validation preprocessing without changing full_data.
image_size = train_state_dict.get("config", {}).get("image_size", 224)
calibration_data = copy.copy(full_data)
calibration_data.transform = transforms.Compose([
    transforms.Resize(round(image_size * 256 / 224)),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

# Sample representative training images reproducibly across the dataset.
calibration_samples = 100
batch_size = 4
indices = torch.randperm(
    len(calibration_data), generator=torch.Generator().manual_seed(0)
)[:calibration_samples].tolist()
calibration_loader = DataLoader(
    Subset(calibration_data, indices),
    batch_size=batch_size,
    shuffle=False,
    drop_last=True,
)
example_inputs = (next(iter(calibration_loader))[0].cpu(),)

# Export the original forward so VGG's adaptive pooling and flatten are retained.
# Export fixes the input shape here: use batch_size=4 and image_size at inference.
float_model = copy.deepcopy(model).cpu().eval()
with torch.no_grad():
    exported_model = torch.export.export(float_model, example_inputs).module()

# The default x86 configuration uses static 8-bit activations and INT8 weights.
quantizer = X86InductorQuantizer()
quantizer.set_global(get_default_x86_inductor_quantization_config())
prepared_model = prepare_pt2e(exported_model, quantizer)

# Calibrate the inserted observers. Eval mode was set before export;
# do not call the earlier quantize_calib helper on this exported model.
with torch.no_grad():
    for images, _ in calibration_loader:
        prepared_model(images.cpu())

quantized_model = convert_pt2e(prepared_model)
qm = quantized_model

# Lower quantized graph patterns into optimized CPU kernels.
# The first invocation includes compilation time; exclude it from benchmarks.
os.environ["TORCHINDUCTOR_FREEZING"] = "1"
with torch.no_grad():
    optimized_model = torch.compile(quantized_model)
    logits = optimized_model(*example_inputs)

print(f"Calibrated with {len(calibration_loader) * batch_size} images")
print(f"Output shape: {tuple(logits.shape)}")
# Evaluate FP32 and optimized_model on separate held-out data to measure accuracy.


/home/mrcorcoran/courses/ece690d/project/ece690dF26_mrc_project/.venv/lib/python3.12/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)


Calibrated with 100 images
Output shape: (4, 100)


In [6]:
# Run after cell 6. Paths are relative to this notebook kernel's working directory.
from pathlib import Path
import importlib.metadata

DEFAULT_SAVE_PATH = Path("runs/vgg16-int8")
DEFAULT_SAVE_PATH.mkdir(parents=True, exist_ok=True)
QUANTIZED_MODEL_PATH = DEFAULT_SAVE_PATH / "quantized_model.pt2"
PRUNING_CHECKPOINT_PATH = DEFAULT_SAVE_PATH / "pruning_checkpoint.pt"

# Save the portable converted graph, rather than the compiled Python wrapper.
# Reload with torch.export.load(), not torch.load().
with torch.no_grad():
    quantized_program = torch.export.export(quantized_model, example_inputs)
torch.export.save(quantized_program, QUANTIZED_MODEL_PATH)

# Preserve the original FP32 parameters for standard nn.utils.prune experiments.
# These are NOT dequantized INT8 weights. After pruning, recalibrate and quantize.
# Exclude optimizer state to keep this checkpoint focused on inference/pruning.
pruning_checkpoint = {
    "format_version": 1,
    "architecture": "vgg16",
    "num_classes": int(float_model.classifier[-1].out_features),
    "model": float_model.state_dict(),
    "classes": train_state_dict.get("classes", []),
    "training_config": train_state_dict.get("config", {}),
    "quantization": {
        "workflow": "torchao_pt2e",
        "backend": "x86_inductor",
        "config": "get_default_x86_inductor_quantization_config",
        "quantized_model_file": QUANTIZED_MODEL_PATH.name,
        "input_shape": list(example_inputs[0].shape),
        "input_dtype": str(example_inputs[0].dtype),
        "calibration_indices": indices,
        "calibration_batch_size": batch_size,
        "calibration_drop_last": True,
        "data_root": str(DATA_ROOT_PATH),
        "dataset": "CIFAR100",
        "dataset_train": True,
        "image_size": image_size,
        "resize_size": round(image_size * 256 / 224),
        "normalize_mean": [0.485, 0.456, 0.406],
        "normalize_std": [0.229, 0.224, 0.225],
    },
    "environment": {
        "torch": str(torch.__version__),
        "torchvision": importlib.metadata.version("torchvision"),
        "torchao": importlib.metadata.version("torchao"),
    },
}
torch.save(pruning_checkpoint, PRUNING_CHECKPOINT_PATH)

# Verify graph deserialization and its outputs before using it elsewhere.
reloaded_quantized_model = torch.export.load(QUANTIZED_MODEL_PATH).module()
with torch.no_grad():
    torch.testing.assert_close(
        reloaded_quantized_model(*example_inputs),
        quantized_model(*example_inputs),
    )

print(f"Quantized graph: {QUANTIZED_MODEL_PATH.resolve()}")
print(f"FP32 pruning checkpoint: {PRUNING_CHECKPOINT_PATH.resolve()}")
print("Quantized save/load output check passed.")


Quantized graph: /home/mrcorcoran/courses/ece690d/project/ece690dF26_mrc_project/misc/runs/vgg16-int8/quantized_model.pt2
FP32 pruning checkpoint: /home/mrcorcoran/courses/ece690d/project/ece690dF26_mrc_project/misc/runs/vgg16-int8/pruning_checkpoint.pt
Quantized save/load output check passed.


### Load these artifacts in another notebook

Use the same PyTorch/TorchAO versions recorded in the checkpoint. Set `SAVE_PATH`
to the saved directory; use an absolute path if the new kernel has a different
working directory. Rerunning the save cell replaces the files at that path.

Load the **quantized graph** for inference or custom graph/quantized-weight experiments:

```python
from pathlib import Path
import torch
import torchao.quantization.pt2e  # Register the PT2E quantization operators.

SAVE_PATH = Path("runs/vgg16-int8")
quantized_model = torch.export.load(SAVE_PATH / "quantized_model.pt2").module()
checkpoint = torch.load(
    SAVE_PATH / "pruning_checkpoint.pt", map_location="cpu", weights_only=True
)
print(checkpoint["environment"])

# Supply real, normalized images matching checkpoint["quantization"]["input_shape"].
# Recompile on this machine when optimized inference is needed:
# with torch.no_grad():
#     optimized_model = torch.compile(quantized_model)
#     logits = optimized_model(images)
```

For **standard PyTorch pruning**, reconstruct the original FP32 module:

```python
from torchvision.models import vgg16

pruning_model = vgg16(weights=None, num_classes=checkpoint["num_classes"])
pruning_model.load_state_dict(checkpoint["model"])
pruning_model.eval()

# Example experiment (run when ready):
# import torch.nn.utils.prune as prune
# prune.l1_unstructured(pruning_model.features[0], name="weight", amount=0.20)
# prune.remove(pruning_model.features[0], "weight")  # Make the zeros permanent.
```

The `.pt` checkpoint contains the **original FP32 weights**, while the `.pt2`
file contains the converted quantized graph. Standard `nn.utils.prune` does not
apply directly to PT2E's exported INT8 graph as if it were an eager VGG module.
The FP32 example is a **prune → recalibrate → quantize** experiment, not pruning
after quantization. True post-quantization pruning requires custom handling of
the graph's quantized weights and their scales/zero points. Keep the `.pt2`
artifact as the quantized baseline for those experiments.

References: [PT2E serialization](https://docs.pytorch.org/ao/stable/pt2e_quantization/pt2e_quant_ptq.html),
[PyTorch pruning](https://docs.pytorch.org/tutorials/intermediate/pruning_tutorial.html).
